# 3. Feature Engineering Pipeline
Deep dive into pipeline structure, transformed features, and validation behavior.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from src.config.core import config
from src.pipeline import pipe
from src.processing.data_manager import load_dataset, load_pipeline, model_file_name, save_pipeline

sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent


In [ ]:
data = load_dataset(config.app_config.training_data_file)
x = data.drop(columns=[config.ml_config.target])
y = data[config.ml_config.target].astype(int)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y,
    test_size=config.ml_config.test_size,
    random_state=config.ml_config.random_state,
    stratify=y
)

print('Train:', x_train.shape, 'Valid:', x_valid.shape)


In [ ]:
step_summary = [(i + 1, name, type(step).__name__) for i, (name, step) in enumerate(pipe.steps)]
step_summary


In [ ]:
pipe.fit(x_train, y_train)
preds = pipe.predict(x_valid)
proba = pipe.predict_proba(x_valid)[:, 1]

metrics = {
    'accuracy': accuracy_score(y_valid, preds),
    'precision': precision_score(y_valid, preds),
    'recall': recall_score(y_valid, preds),
    'f1': f1_score(y_valid, preds),
    'roc_auc': roc_auc_score(y_valid, proba),
}
metrics


In [ ]:
cm = confusion_matrix(y_valid, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.ml_config.random_state)
cv_auc = cross_val_score(pipe, x, y, cv=cv, scoring='roc_auc')
print('CV ROC AUC scores:', np.round(cv_auc, 4))
print('Mean CV ROC AUC :', round(cv_auc.mean(), 4))


In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(proba[y_valid.values == 0], bins=30, stat='density', alpha=0.5, label='target=0')
sns.histplot(proba[y_valid.values == 1], bins=30, stat='density', alpha=0.5, label='target=1')
plt.title('Predicted Probability by True Class')
plt.legend()
plt.show()


In [ ]:
save_pipeline(pipe)
reloaded = load_pipeline(model_file_name())
reloaded_preds = reloaded.predict(x_valid.head(20))
print('Reloaded model prediction sample:', reloaded_preds[:10])
